In [1]:
import math
import numpy as np
import random

In [2]:
class Value:

    def __init__(self, data, _children=(), _op=''):
        self.data = data
        self.grad = 0
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op

    def __repr__(self):
        return f'Value(data={self.data})'

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += (other.data) * out.grad
            other.grad += (self.data) * out.grad
        out._backward = _backward
        
        return out

    def __pow__(self, other):
        assert isinstance (other, (int, float))
        out = Value(self.data ** other, (self, ), f'**{other}')

        def _backward():
            self.grad += other * (self.data ** (other - 1)) * out.grad
        out._backward = _backward

        return out
    
    def __rmul__(self, other):
        return self * other

    def __truediv__(self, other):
        return self * (other**-1)

    def __neg__(self):
        return -1 * self

    def __sub__(self, other):
        return self + (-other)
    
    def tanh(self):
        x = self.data
        t = (math.exp(2 * x ) - 1)/(math.exp(2 * x ) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad
        out._backward = _backward
        
        return out
    
    def exp(self):
        x = self.data
        t = math.exp(x)
        out = Value(t, (self, ), 'exp')

        def _backward():
            self.grad += t * out.grad
        out._backward = _backward
        
        return out

    def backward(self):
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        
        self.grad = 1.0
        for node in reversed(topo):
            node._backward()


In [3]:
#inputs
x1 = Value(2.0)
x2 = Value(0.0)

#weights
w1 = Value(-3.0)
w2 = Value(1.0)

#bias
b = Value(6.8813735870195432)

x1w1 = w1 * x1
x2w2 = w2 * x2
x1w1xx2w2 = x1w1 + x2w2
n = x1w1xx2w2 + b
#---
e = (2 * n).exp()
o = (e-1)/(e+1)
#o = n.tanh()
#----
o.backward()

In [57]:
class Neuron:

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1, 1)) for _ in range(nin)]
        self.b = Value(random.uniform(-1, 1))

    def __call__(self, x):
        add = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        out = add.tanh()
        return out

    def parameters(self):
        return self.w + [self.b]

class Layer:

    def __init__(self, nin, nout):
        #nin - input per neuron
        #nout - no. of neuron
        self.neurons = [Neuron(nin) for _ in range(nout)]
        #(3, 4) - 3 input per neuron and no of neuron is 4

    def __call__(self, x):
        outs = [n(x) for n in self.neurons]
        #runs the input through every neuron
        return outs[0] if len(outs) == 1 else outs

    def parameters(self):
        return [p for neuron in self.neurons for p in neuron.parameters()]

class MLP:

    def __init__(self, nin, nouts):
        sz = [nin] + nouts
        self.layers = [Layer(sz[i], sz[i+1]) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

In [58]:
x = [2.0, 3.0, -1.0]
n = MLP(3, [4, 4, 1])
n(x)

Value(data=-0.7742720545072028)

In [59]:
xs = [
    [2.0, 3.0, -1.0],
    [3.0, -1.0, 0.5],
    [0.5, 0.5, 1.0],
    [1.0, 1.0, -1.0]
]

ys = [1.0,-1.0, -1.0, 1.0]
ypred = [n(x) for x in xs]
ypred

[Value(data=-0.7742720545072028),
 Value(data=-0.9116256996387859),
 Value(data=-0.8936596795043241),
 Value(data=-0.7985653251110911)]

In [60]:
for k in range(1000):  # training loop
    # 1. Forward pass
    ypred = [n(x) for x in xs]
    loss = sum(((yout - ygt)**2 for ygt, yout in zip(ys, ypred)), Value(0))

    # 2. Zero gradients 
    for p in n.parameters():
        p.grad = 0.0

    # 3. Backward pass
    loss.backward()

    # 4. Update weights
    for p in n.parameters():
        p.data -= 0.01 * p.grad
    if k%100 == 0:
        print(k, loss.data)

0 6.401996832824633
100 0.0405550685192825
200 0.01858691223412908
300 0.01180032313716571
400 0.008563378818355722
500 0.006685397378597593
600 0.0054652993787242185
700 0.004611710652053285
800 0.003982477439463458
900 0.003500217550015683
